# Tasks

## P 5.1: Download the CarSharing dataset from Canvas. Train a deep neural network to predict the 'demand' column. Tune the network hyperparameters to find the best set of hyperparameters that produce the most accurate results. Evaluate the model using a five-fold cross-validation method and calculate all regression evaluation metrics (15%).

In [1]:
############# WRITE YOUR CODE IN THIS CELL (IF APPLICABLE)  ####################
############# WRITE YOUR CODE IN THIS CELL (IF APPLICABLE) ####################

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv("csv/CarSharing.csv")

# -------------------------------
# Handle datetime features
# -------------------------------
for col in df.columns:
    if df[col].dtype == 'object':
        try:
            df[col] = pd.to_datetime(df[col])
            df[col + "_hour"] = df[col].dt.hour
            df[col + "_day"] = df[col].dt.day
            df[col + "_month"] = df[col].dt.month
            df.drop(columns=[col], inplace=True)
        except:
            pass

# Keep numeric columns only
df = df.select_dtypes(include=['int64', 'float64'])
df = df.dropna()

# -------------------------------
# Features and target
# -------------------------------
X = df.drop(columns=['demand'])
y = df['demand']

# -------------------------------
# Feature scaling
# -------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -------------------------------
# Model builder function
# -------------------------------
def build_model(input_dim, lr=0.001, neurons=64, dropout=0.2):
    model = Sequential([
        Dense(neurons, activation='relu', input_shape=(input_dim,)),
        Dropout(dropout),
        Dense(neurons // 2, activation='relu'),
        Dense(1)
    ])
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='mse'
    )
    return model

# -------------------------------
# Hyperparameter options
# -------------------------------
learning_rates = [0.001, 0.0005]
neurons_list = [64, 128]
epochs = 50
batch_size = 32

# -------------------------------
# 5-Fold Cross Validation
# -------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for lr in learning_rates:
    for neurons in neurons_list:
        fold = 1
        mae_list, mse_list, rmse_list, r2_list = [], [], [], []

        for train_idx, test_idx in kf.split(X_scaled):
            X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            model = build_model(
                input_dim=X_train.shape[1],
                lr=lr,
                neurons=neurons
            )

            model.fit(
                X_train, y_train,
                epochs=epochs,
                batch_size=batch_size,
                verbose=0
            )

            y_pred = model.predict(X_test).flatten()

            mae_list.append(mean_absolute_error(y_test, y_pred))
            mse_list.append(mean_squared_error(y_test, y_pred))
            rmse_list.append(np.sqrt(mean_squared_error(y_test, y_pred)))
            r2_list.append(r2_score(y_test, y_pred))

            fold += 1

        results.append({
            "Learning Rate": lr,
            "Neurons": neurons,
            "MAE": np.mean(mae_list),
            "MSE": np.mean(mse_list),
            "RMSE": np.mean(rmse_list),
            "R2": np.mean(r2_list)
        })

# -------------------------------
# Display Results
# -------------------------------
results_df = pd.DataFrame(results)
print(results_df)



KeyboardInterrupt: 